# M4.3 · RNN ve BPTT

Zaman içinde paylaşılan ağırlıklar. Skaler bir RNN'i üç adım ileri ve geri (backpropagation through time) elle çalıştırıyor, saf Python'da vektör RNN'in BPTT'sini yazıyor, NumPy'da CS231n'in `rnn_forward/backward`'ını kuruyor, PyTorch'ta `nn.RNN` ile aynı sayıları veren kendi RNN'ini yazıp gradyanın zamanda nasıl söndüğünü ölçüyor ve en sonda doğrusal bir rekürsiyonu paralel scan ile $O(\log T)$ adımda hesaplıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m4-3-rnn.html](https://m-gonul.github.io/pytorch-lab/m4-3-rnn.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m4_3_rnn.py
CHECKS = {
 "b": "import math\nimport random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def hand():\n        xs = [[1.0], [0.0], [0.0]]\n        hs = need(ns, \"rnn_forward\")(xs, [0.0], [[1.0]], [[0.5]], [0.0])\n        close([h[0] for h in hs], [0.761594, 0.363399, 0.179726], atol=1e-5)\n        dWx, dWh, db, dh0 = need(ns, \"rnn_backward\")(xs, hs, [0.0], [[1.0]], [[0.5]], [[0.0], [0.0], [1.0]])\n        close(dWx[0][0], 0.088185, atol=1e-5, what=\"∂L/∂u\")\n    R.test(\"elle örnek\", hand)\n\n    def num():\n        rng = random.Random(0)\n        D, H, T = 2, 3, 4\n        xs = [[rng.gauss(0, 1) for _ in range(D)] for _ in range(T)]\n        h0 = [rng.gauss(0, 0.5) for _ in range(H)]\n        Wx = [[rng.gauss(0, 0.7) for _ in range(D)] for _ in range(H)]\n        Wh = [[rng.gauss(0, 0.7) for _ in range(H)] for _ in range(H)]\n        b = [rng.gauss(0, 0.3) for _ in range(H)]\n        C = [[rng.gauss(0, 1) for _ in range(H)] for _ in range(T)]\n        L = lambda Wx_, Wh_, h0_: sum(c * v for ct, ht in zip(C, need(ns, \"rnn_forward\")(xs, h0_, Wx_, Wh_, b)) for c, v in zip(ct, ht))\n        hs = need(ns, \"rnn_forward\")(xs, h0, Wx, Wh, b)\n        dWx, dWh, db, dh0 = need(ns, \"rnn_backward\")(xs, hs, h0, Wx, Wh, C)\n        e = 1e-6\n        for i in range(H):\n            for k in range(H):\n                P = [r[:] for r in Wh]; M = [r[:] for r in Wh]; P[i][k] += e; M[i][k] -= e\n                close(dWh[i][k], (L(Wx, P, h0) - L(Wx, M, h0)) / (2 * e), rtol=1e-5, atol=1e-7, what=f\"dWh[{i}][{k}]\")\n        for k in range(H):\n            P, M = h0[:], h0[:]; P[k] += e; M[k] -= e\n            close(dh0[k], (L(Wx, Wh, P) - L(Wx, Wh, M)) / (2 * e), rtol=1e-5, atol=1e-7, what=f\"dh0[{k}]\")\n    R.test(\"D = 2, H = 3, T = 4: sayısal gradyanla aynı\", num)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"yalnız zaman döngüsü\", lambda: (max_loops(src, 1, func=\"rnn_forward\"), max_loops(src, 1, func=\"rnn_backward\")))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n    N, T, D, H = 3, 5, 4, 6\n    x = rng.normal(size=(N, T, D)); h0 = rng.normal(size=(N, H)) * 0.5\n    Wx, Wh, b = rng.normal(size=(D, H)) * 0.5, rng.normal(size=(H, H)) * 0.5, rng.normal(size=H) * 0.1\n    dout = rng.normal(size=(N, T, H))\n\n    def fw():\n        h, _ = need(ns, \"rnn_forward\")(x, h0, Wx, Wh, b)\n        prev = h0\n        for t in range(T):\n            prev = np.tanh(x[:, t] @ Wx + prev @ Wh + b)\n            close(h[:, t], prev, what=f\"t = {t}\")\n    R.test(\"rnn_forward\", fw)\n\n    def bw():\n        _, cache = need(ns, \"rnn_forward\")(x, h0, Wx, Wh, b)\n        dx, dh0, dWx, dWh, db = need(ns, \"rnn_backward\")(dout, cache)\n        f = lambda: (need(ns, \"rnn_forward\")(x, h0, Wx, Wh, b)[0] * dout).sum()\n        e = 1e-6\n        for arr, g, name in ((x, dx, \"dx\"), (h0, dh0, \"dh0\"), (Wx, dWx, \"dWx\"), (Wh, dWh, \"dWh\"), (b, db, \"db\")):\n            for idx in list(np.ndindex(arr.shape))[::max(1, arr.size // 6)]:\n                old = arr[idx]; arr[idx] = old + e; p = f(); arr[idx] = old - e; m = f(); arr[idx] = old\n                close(g[idx], (p - m) / (2 * e), rtol=1e-5, atol=1e-7, what=f\"{name}{idx}\")\n    R.test(\"rnn_backward sayısal gradyanla aynı (tüm gradyanlar)\", bw)\n",
 "d": "import torch\nfrom torch import nn\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    g = torch.Generator().manual_seed(0)\n\n    def eq():\n        C = need(ns, \"MyRNN\", \"sınıf\")\n        ref = nn.RNN(4, 6, batch_first=True).double()\n        my = C(4, 6).double()\n        my.load_state_dict(ref.state_dict())\n        x = torch.randn(3, 7, 4, generator=g, dtype=torch.float64)\n        h0 = torch.randn(1, 3, 6, generator=g, dtype=torch.float64)\n        o1, h1 = my(x, h0); o2, h2 = ref(x, h0)\n        close(o1, o2, what=\"out\"); close(h1, h2, what=\"h_n\")\n        w = torch.randn_like(o1)\n        (o1 * w).sum().backward(); (o2 * w).sum().backward()\n        for (n, a), (_, b) in zip(my.named_parameters(), ref.named_parameters()):\n            close(a.grad, b.grad, what=f\"∂{n}\")\n        o3, _ = my(x)\n        close(o3, ref(x)[0], what=\"h0 = None\")\n    R.test(\"MyRNN = nn.RNN (çıktı + gradyanlar)\", eq)\n\n    def van():\n        torch.manual_seed(0)\n        rnn = nn.RNN(8, 32, batch_first=True)\n        x = torch.randn(4, 50, 8, generator=g)\n        n = need(ns, \"input_grad_norms\")(rnn, x)\n        ensure(len(n) == 50, f\"50 değer bekleniyordu, {len(n)} geldi\")\n        ensure(n[0] < 1e-3 * n[-1], f\"gradyan geçmişe doğru sönmeli: ilk {n[0]:.2e}, son {n[-1]:.2e}\")\n        R.table(\"‖∂(Σh₅₀)/∂x_t‖ — varsayılan başlatmalı nn.RNN\", [\"t\", \"norm\"], [[str(t + 1), f\"{n[t]:.2e}\"] for t in (49, 44, 39, 29, 19, 0)])\n    R.test(\"gradyan zamanda sönüyor\", van)\n",
 "e": "import torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef seq(a, b):\n    h, out = torch.zeros_like(b[0]), []\n    for t in range(a.shape[0]):\n        h = a[t] * h + b[t]; out.append(h)\n    return torch.stack(out)\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"yalnız offset döngüsü\", lambda: max_loops(src, 1, func=\"linear_scan\", why=\"zaman üzerinde döngü olmamalı\"))\n    g = torch.Generator().manual_seed(0)\n\n    def eq():\n        for T in (1, 7, 64, 100):\n            a = torch.rand(T, 3, 5, generator=g, dtype=torch.float64) * 0.9 + 0.05\n            b = torch.randn(T, 3, 5, generator=g, dtype=torch.float64)\n            close(need(ns, \"linear_scan\")(a, b), seq(a, b), what=f\"T = {T}\")\n    R.test(\"sıralı rekürsiyonla aynı (T = 1, 7, 64, 100)\", eq)\n\n    def gr():\n        a = (torch.rand(33, 4, generator=g, dtype=torch.float64) * 0.9).requires_grad_(True)\n        b = torch.randn(33, 4, generator=g, dtype=torch.float64).requires_grad_(True)\n        need(ns, \"linear_scan\")(a, b).sum().backward()\n        ga, gb = a.grad.clone(), b.grad.clone()\n        a.grad = b.grad = None\n        seq(a, b).sum().backward()\n        close(ga, a.grad, what=\"∂a\"); close(gb, b.grad, what=\"∂b\")\n    R.test(\"autograd scan içinden de doğru akıyor\", gr)\n    if \"linear_scan\" not in ns:\n        return\n    rows = []\n    for T in (1024, 8192):\n        a = torch.rand(T, 64, generator=g) * 0.99; b = torch.randn(T, 64, generator=g)\n        t1, t2 = bench(lambda: seq(a, b), repeat=3, number=1), bench(lambda: ns[\"linear_scan\"](a, b), repeat=3)\n        rows.append([str(T), fmt_time(t1), fmt_time(t2), f\"{t1 / t2:,.1f}×\"])\n    R.table(\"Doğrusal rekürsiyon, 64 kanal\", [\"T\", \"sıralı (T adım)\", \"scan (log₂T tur)\", \"kazanç\"], rows,\n            note=\"Scan O(T log T) iş yapar ama derinliği O(log T). GPU'da Blelloch (O(T) iş) ya da çip-içi kernel'lar kullanılır.\")\n",
 "x1": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(0)\n        a = rng.normal(3, 5, (4, 10))\n        y = need(ns, \"ln\")(a, np.ones(10), np.zeros(10))\n        close((y.mean(-1), y.std(-1)), (np.zeros(4), np.ones(4)), atol=1e-4, what=\"LN: örnek başına\")\n        X = rng.normal(size=(200, 16, 8))\n        H = 64\n        rows = []\n        res = {}\n        for scale in (0.5, 1.0, 2.0):\n            Wx = rng.normal(0, 1 / np.sqrt(8), (8, H))\n            Wh = rng.normal(0, scale / np.sqrt(H), (H, H))\n            for name, lnp in ((\"RNN\", None), (\"LN-RNN\", (np.ones(H), np.zeros(H)))):\n                Hs = need(ns, \"rnn_run\")(X, Wx, Wh, np.zeros(H), lnp)\n                sat = (np.abs(Hs[-50:]) > 0.97).mean()\n                res[(scale, name)] = (np.abs(Hs[-50:]).mean(), sat)\n                rows.append([scale, name, f\"{np.abs(Hs[-50:]).mean():.3f}\", f\"{sat:.3f}\"])\n        ensure(res[(2.0, \"RNN\")][1] > 3 * res[(2.0, \"LN-RNN\")][1], \"LN doymayı engeller\")\n        ensure(abs(res[(0.5, \"LN-RNN\")][0] - res[(2.0, \"LN-RNN\")][0]) < 0.05, \"LN-RNN ölçekten bağımsız\")\n        R_.table(\"200 adımlık RNN (64 birim): son 50 adımda gizli durum\", [\"W_h ölçeği\", \"model\", \"ort. |h|\", \"doymuş oran (|h| > 0.97)\"], rows,\n                 note=\"Düz RNN'de rekürent ağırlık ölçeği her şeyi belirler: küçükse sinyal söner, büyükse tanh doyar (türev ≈ 0, gradyan akmaz). LN her adımda ön-aktivasyonu sıfır ortalamalı, birim varyanslı yapar: dinamik ağırlık ölçeğinden bağımsızlaşır, tanh doğrusal bölgesinde kalır. Géron'un 'kararsız gradyanlar' çözümlerinden biri (diğeri: rekürent dropout, gradyan kırpma).\")\n    R_.test(\"LayerNorm RNN\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(0)\n        x = rng.normal(size=60)\n        L = lambda w: 0.5 * (need(ns, \"forward\")(w, 1.0, x)[-1] - 0.3) ** 2\n        g = need(ns, \"grad_w\")(0.9, 1.0, x, 0.3)\n        close(g, (L(0.9 + 1e-7) - L(0.9 - 1e-7)) / 2e-7, rtol=1e-5, what=\"tam BPTT = sayısal\")\n        rows = []\n        for w in (0.5, 0.9, 0.98):\n            full = ns[\"grad_w\"](w, 1.0, x, 0.3)\n            r = [w]\n            for k in (1, 5, 20, 50):\n                r.append(f\"{abs(ns['grad_w'](w, 1.0, x, 0.3, k) - full) / abs(full):.3f}\")\n            rows.append(r)\n        ensure(float(rows[0][3]) < 1e-4 and float(rows[2][3]) > 0.05, \"w küçükse kısa pencere yeter, w ≈ 1'de yetmez\")\n        R_.table(\"Skaler doğrusal RNN, T = 60: kesilmiş BPTT gradyanının göreli hatası\", [\"w\", \"k = 1\", \"k = 5\", \"k = 20\", \"k = 50\"], rows,\n                 note=\"t adım önceki katkı w^t ile söner: |w| < 1 ve küçükse birkaç adım yeter, |w| → 1 (uzun bellek) olduğunda kesme yanlılığı büyür — model, pencereden uzun bağımlılıkları öğrenemez. Pratikte (Williams & Peng 1990) uzun diziler parçalara bölünür, gizli durum parçalar arasında taşınır (gradyan taşınmaz): ileri yön tam bellek, geri yön k adım.\")\n    R_.test(\"kesilmiş BPTT\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        L0, g0 = need(ns, \"loss_grad\")(1.5, -0.5, 30, 0.5, 0.0)\n        Lp = ns[\"loss_grad\"](1.5 + 1e-6, -0.5, 30, 0.5, 0.0)[0]\n        close(g0, (Lp - L0) / 1e-6, rtol=1e-3, what=\"BPTT gradyanı\")\n        rows = []\n        res = {}\n        for T in (5, 10, 20, 40):\n            wc = need(ns, \"find_wall\")(-1.0, T, 0.5, 2.5, 4.0)\n            far = abs(ns[\"loss_grad\"](wc - 0.3, -1.0, T, 0.5, 2.0)[1])\n            near = max(abs(ns[\"loss_grad\"](wc - dd, -1.0, T, 0.5, 2.0)[1]) for dd in [0.0] + [10.0 ** -k for k in range(2, 15)])\n            jump = abs(need(ns, \"step\")(wc, near, 0.01) - wc)\n            jc = abs(ns[\"step\"](wc, near, 0.01, 1.0) - wc)\n            res[T] = near\n            rows.append([T, f\"{wc:.4f}\", f\"{far:.2e}\", f\"{near:.2e}\", f\"{jump:.2e}\", f\"{jc:.2e}\"])\n        ensure(res[40] > 100 * res[10] and abs(float(rows[3][1]) - 1.5 / np.tanh(0.5)) < 0.01, \"duvarda gradyan T ile patlar; duvar ≈ 1.5/tanh(0.5)\")\n        R_.table(\"h_t = w·tanh(h_(t−1)) − 1, h₀ = 0.5, hedef 2\", [\"T\", \"duvar w*\", \"|∂L/∂w| (w* − 0.3)\", \"duvarda maks. |∂L/∂w|\", \"SGD adımı (η = 0.01)\", \"kırpılmış adım (|g| ≤ 1)\"], rows,\n                 note=\"w* ≈ 1.5/tanh(0.5) ≈ 3.25'te kararsız sabit nokta h₀'ın üzerinden geçer: altında yörünge negatif çekere, üstünde pozitif çekere gider. Duvara yakın gradyan uzun dizilerde astronomikleşir — tek bir kırpmasız adım parametreyi bütün anlamlı aralığın dışına atar. Kırpma, gradyan yönünü koruyup adımı sınırlar (Pascanu ve ark. Şekil 6).\")\n    R_.test(\"uçurum ve kırpma\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m4_3_rnn as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Bir RNN her zaman adımında aynı ağırlıklarla bir gizli durumu günceller:

$$
h_t=\tanh\big(x_tW_x+h_{t-1}W_h+b\big),\qquad t=1..T .
$$

Zamanda açıldığında bu, $T$ katmanlı ve **ağırlıkları paylaşılan** bir ileri beslemeli ağdır. Backpropagation through time (BPTT) bu açılmış grafikte sıradan backprop'tur:

$$
\frac{\partial L}{\partial h_{t-1}}=\Big(\frac{\partial L}{\partial h_t}\odot(1-h_t^2)\Big)W_h^\top+\frac{\partial L_{t-1}}{\partial h_{t-1}},\qquad
\frac{\partial L}{\partial W_h}=\sum_t h_{t-1}^\top\Big(\frac{\partial L}{\partial h_t}\odot(1-h_t^2)\Big).
$$

Uzak geçmişe giden gradyan, $\prod_t W_h^\top\text{diag}(1-h_t^2)$ çarpımını içerir. Bu çarpım büyük $T$'de sıfıra (sönme) ya da sonsuza (patlama) gider. Patlamaya çare gradient clipping; sönmeye çare kapılı hücreler (M4.4).

Tanh olmasaydı rekürsiyon doğrusal olurdu: $h_t=a_t\odot h_{t-1}+b_t$. Bunun özel bir yanı var: adımlar birleşmeli bir işlemle birleşir ve associative scan ile $O(\log T)$ paralel adımda hesaplanır. Mamba gibi state-space modellerin hızı buradan gelir.

## 1 · Elle

İleri geçişte tek girdi ilk adımda; sonrasında durum her adımda yaklaşık yarıya iniyor ($w=0.5$).

Geri geçişte gradyan her adımda $w(1-h^2)<0.5$ ile çarpılıyor. Üç adımda $0.21$'e düştü; 30 adımda $10^{-10}$ civarında olurdu. RNN'in uzak geçmişi "unutması" bu çarpımdan.

$\partial L/\partial u$'da $u$ üç adımda da kullanıldı ama $x_2=x_3=0$ olduğu için yalnız ilk adımın katkısı kaldı.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Skaler RNN $h_t=\tanh(w\,h_{t-1}+u\,x_t)$, $w=0.5$, $u=1$, $h_0=0$, $x=[1,0,0]$, kayıp $L=h_3$**

- `h1`: $h_1$
- `h2`: $h_2$
- `h3`: $h_3$

**BPTT: $\frac{\partial h_t}{\partial h_{t-1}}=w(1-h_t^2)$**

- `dh2`: $\partial L/\partial h_2$
- `dh1`: $\partial L/\partial h_1$
- `du`: $\partial L/\partial u$

In [ ]:
cevaplar = dict(
    h1=None,
    h2=None,
    h3=None,
    dh2=None,
    dh1=None,
    du=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"h1":[0.761594,0.002],"h2":[0.363399,0.002],"h3":[0.179726,0.002],"dh2":[0.483849,0.002],"dh1":[0.209976,0.002],"du":[0.088185,0.002]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python; vektörler liste, matrisler satır listesi. **Matris düzeni:** $W_x$ $(H\times D)$, $W_h$ $(H\times H)$, $h_t=\tanh(W_xx_t+W_hh_{t-1}+b)$.

- `rnn_forward(xs, h0, Wx, Wh, b)` → `hs` (uzunluk $T$ liste; $h_1..h_T$).
- `rnn_backward(xs, hs, h0, Wx, Wh, dhs)` — `dhs[t]`: kaybın $h_{t+1}$'e **doğrudan** gradyanı. → `(dWx, dWh, db, dh0)`. Zamanda geriye yürü, dolaylı gradyanı her adımda ekle.

In [ ]:
%%writefile stage_b.py
import math


def rnn_forward(xs, h0, Wx, Wh, b):
    ...


def rnn_backward(xs, hs, h0, Wx, Wh, dhs):
    """→ (dWx, dWh, db, dh0)"""
    ...


hs = rnn_forward([[1.0], [0.0], [0.0]], [0.0], [[1.0]], [[0.5]], [0.0])
print(hs)                                                     # [[0.7616], [0.3634], [0.1797]]
print(rnn_backward([[1.0], [0.0], [0.0]], hs, [0.0], [[1.0]], [[0.5]], [[0.0], [0.0], [1.0]]))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

CS231n arayüzü, batch'li, satır vektörleri: `x (N, T, D)`, `h0 (N, H)`, `Wx (D, H)`, `Wh (H, H)`, `b (H,)`.

- `rnn_forward(x, h0, Wx, Wh, b)` → `(h, cache)`, `h: (N, T, H)`.
- `rnn_backward(dh, cache)` → `(dx, dh0, dWx, dWh, db)`; `dh: (N, T, H)` her adımın doğrudan gradyanı.

Yalnız zaman döngüsü (adım içinde döngü yok).

In [ ]:
%%writefile stage_c.py
import numpy as np


def rnn_forward(x, h0, Wx, Wh, b):
    ...


def rnn_backward(dh, cache):
    """→ (dx, dh0, dWx, dWh, db)"""
    ...

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

- `MyRNN(input_size, hidden_size)` — `nn.Module`, `batch_first=True` davranışı; parametre adları `nn.RNN` ile aynı: `weight_ih_l0 (H, D)`, `weight_hh_l0 (H, H)`, `bias_ih_l0`, `bias_hh_l0`. `forward(x, h0=None)` → `(out: (N, T, H), h_n: (1, N, H))`. Böylece `my.load_state_dict(torch_rnn.state_dict())` çalışır.
- `input_grad_norms(rnn, x)` → uzunluğu $T$ liste: $\lVert\partial\,(\sum h_T)/\partial x_t\rVert$, $t=1..T$ (her örnek ve özellik üzerinden norm).

Kontrol `nn.RNN` ile çıktı ve gradyan eşitliğini, sonra 50 adımlık bir dizide gradyanın geçmişe doğru sönmesini gösterir.

In [ ]:
%%writefile stage_d.py
import torch
from torch import nn


class MyRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        ...

    def forward(self, x, h0=None):
        ...


def input_grad_norms(rnn, x):
    """→ [‖∂(Σ h_T)/∂x_t‖ for t]"""
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Doğrusal rekürsiyon $h_t=a_t\odot h_{t-1}+b_t$ ($h_0=0$), `a, b: (T, …)`.

Her adım bir afin dönüşüm $h\mapsto a\,h+b$; iki adımın bileşimi yine afin: $(a_2,b_2)\circ(a_1,b_1)=(a_2a_1,\;a_2b_1+b_2)$. **Hillis–Steele** scan: `offset = 1, 2, 4, …` için her $t\ge\text{offset}$'te

```text
B[t] ← A[t]·B[t−offset] + B[t]
A[t] ← A[t]·A[t−offset]
```

(sağ taraflar güncellemeden **önceki** değerler). $\lceil\log_2T\rceil$ turdan sonra $B[t]=h_t$.

- `linear_scan(a, b)` → `h` (aynı shape). Yalnız `offset` döngüsü ($\log T$ tur); zaman üzerinde döngü yok.

In [ ]:
%%writefile stage_e.py
import torch


def linear_scan(a, b):
    """h_t = a_t·h_{t−1} + b_t, h_0 = 0. a, b: (T, …). Yalnız offset döngüsü."""
    ...


a = torch.full((8,), 0.5)
b = torch.tensor([1.0, 0, 0, 0, 0, 0, 0, 0])
print(linear_scan(a, b))     # [1, 0.5, 0.25, …]

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Katman normalizasyonlu RNN

**Katman normalizasyonlu RNN** (Ba, Kiros, Hinton 2016; Géron Bölüm 15): BN zaman adımları arasında zordur (her adım ayrı istatistik, değişken uzunluk); LayerNorm her örneği ve her adımı kendi öznitelikleri üzerinden normalize eder:
$$h_t=\tanh\big(\mathrm{LN}(W_xx_t+W_hh_{t-1})\big),\quad\mathrm{LN}(a)=\gamma\odot\frac{a-\mu(a)}{\sigma(a)}+\beta.$$

NumPy.

- `ln(a, gamma, beta, eps=1e-5)` — son eksende.
- `rnn_run(X, Wx, Wh, b, ln_params=None)` — `X (T, B, d)`; `ln_params=(gamma, beta)` verilirse LN-RNN. Tüm gizli durumlar `(T, B, h)`; zaman döngüsü serbest.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def ln(a, gamma, beta, eps=1e-5):
    ...


def rnn_run(X, Wx, Wh, b, ln_params=None):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Kesilmiş BPTT'nin yanlılığı

**Kesilmiş BPTT** (truncated BPTT): uzun dizilerde gradyan yalnız son k adım geriye yayılır — bellek ve hesap O(k), ama k'dan uzak bağımlılıkların katkısı kaybolur (yanlı tahmin).

Skaler doğrusal RNN $h_t=wh_{t-1}+ux_t$, kayıp $L=\tfrac12(h_T-y)^2$. Gerçek gradyan $\frac{\partial L}{\partial w}=(h_T-y)\sum_{t=1}^{T}w^{T-t}h_{t-1}$.

Saf Python + NumPy.

- `forward(w, u, x)` → `h` listesi (`h[0] = 0`).
- `grad_w(w, u, x, y, k=None)` — `k=None` tam BPTT; `k` verilirse yalnız son k terim.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def forward(w, u, x):
    ...


def grad_w(w, u, x, y, k=None):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Patlayan gradyanlar: kayıp yüzeyindeki duvar ve kırpma

**Patlayan gradyanlar ve uçurum** (Pascanu, Mikolov, Bengio 2013): $h_t=w\tanh(h_{t-1})+b$ gibi bir RNN'de parametre bir çatallanma (bifurcation) noktasını geçince $h_T$ bir çekerden diğerine sıçrar; kayıp yüzeyinde dik bir "duvar" oluşur. Duvarın yakınında gradyan T ile üstel büyür ve tek bir SGD adımı parametreyi çok uzağa fırlatır; norma göre kırpma adım boyunu sınırlar.

NumPy.

- `loss_grad(w, b, T, h0, target)` — $L=(h_T-\text{target})^2$; BPTT ile $\partial L/\partial w$ (zaman döngüsü serbest).
- `find_wall(b, T, h0, lo, hi, iters=60)` — $h_T$'nin işaret değiştirdiği w'yi ikiye bölmeyle bul ($h_T(lo)<0<h_T(hi)$ varsayılır).
- `step(w, g, lr, clip=None)` — bir SGD adımı; `clip` verilirse $|g|\le\text{clip}$.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def loss_grad(w, b, T, h0, target):
    ...


def find_wall(b, T, h0, lo, hi, iters=60):
    ...


def step(w, g, lr, clip=None):
    ...

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Pascanu, Mikolov, Bengio (2013): RNN eğitmenin zorluğu

[Pascanu, Mikolov, Bengio — On the difficulty of training recurrent neural networks (ICML 2013)](https://arxiv.org/abs/1211.5063)

Makalenin sentetik "patolojik" görevlerinden ikisini (toplama problemi ve zamansal sıra problemi, Hochreiter & Schmidhuber 1997) T ∈ {50, 100, 200} için kur; düz tanh RNN'i (a) yalın SGD, (b) norm kırpma, (c) kırpma + makalenin kaybolan gradyan düzenleyicisi Ω (Denk. 9) ile eğit ve başarı oranlarını (Şekil 7) raporla. x3'teki duvar analizini 2 boyutlu gizli durumla tekrarlayıp kayıp yüzeyini çiz.


**Adımlar**

1. Ω'nun gradyanını yaklaşık hesapla: ∂h_{k+1}/∂h_k'yı sabit sayarak (makalenin önerisi).

**Kabul ölçütleri**

- Kırpmasız eğitim uzun T'de ıraksar ya da takılır; kırpma kararlılık sağlar.
- Ω, uzun T'de başarı oranını artırır.

## Makale ödevi: Ba, Kiros, Hinton (2016): Layer Normalization

[Ba, Kiros, Hinton — Layer Normalization (2016)](https://arxiv.org/abs/1607.06450)

x1'deki LN-RNN'i PyTorch'ta bir GRU/LSTM hücresine taşı (makale Ek A: her kapının ön-aktivasyonunu ayrı normalize et) ve karakter düzeyinde dil modellemesinde (Shakespeare, ~1 MB) LN'li ve LN'siz modellerin doğrulama kaybını adım sayısına göre karşılaştır. Makale Bölüm 5'teki yeniden ölçekleme değişmezliği tablosunu (Tablo 1) küçük deneylerle doğrula: ağırlık matrisini ve girdiyi ölçeklemek LN çıktısını değiştirir mi?


**Adımlar**

1. Uzun dizilerde (T = 200) gradyan normlarının zamana göre dağılımını çiz.

**Kabul ölçütleri**

- LN'li model daha hızlı ve daha kararlı yakınsar.
- Değişmezlik tablosu deneylerle uyumlu.

## Kaynaklar

- [CS231n Assignment 3 — RNN captioning (rnn_step_forward/backward)](https://cs231n.github.io/assignments2025/assignment3/) — MIT
- [Dive into Deep Learning §9 Recurrent Neural Networks, §9.7 BPTT](https://d2l.ai/chapter_recurrent-neural-networks/bptt.html) — CC BY-SA 4.0 / MIT-0
- [Pascanu, Mikolov, Bengio — On the difficulty of training RNNs](https://arxiv.org/abs/1211.5063) — arXiv
- [Blelloch — Prefix Sums and Their Applications](https://www.cs.cmu.edu/~guyb/papers/Ble93.pdf) — ©
- [Gu & Dao — Mamba: Linear-Time Sequence Modeling with Selective State Spaces](https://arxiv.org/abs/2312.00752) — arXiv